In [1]:
import random
import numpy as np
import torch
import torch.nn as nn
import torchmetrics
import transformers
import tokenizers
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from collections import namedtuple

device = 'cuda'

/home/damian/New Folder/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def gen_reber():
    reber_text_long = 'B'
    next_letter = ['T', 'P'][random.randint(0, 1)]
    
    rules = {'B':['T1', 'P1'], 'T1':['S1', 'X1'], 'P1':['T2', 'V1'], 'S1':['S1', 'X1'], 'T2': ['T2', 'V1'], 'X1': ['X2', 'S2'], 
             'V1':['V2', 'P2'], 'P2':['X2', 'S2'], 'X2':['T2', 'V1'],'S2':['E', 'E'], 'V2':['E', 'E']}
    node = 'B'
    reber_text_draft = 'B'
    while node != 'E':
        index = random.randint(0, 1)
        node = rules[node][index]
        reber_text_draft += node
    
    reber_text = ''
    for char in reber_text_draft:
        if not char.isdigit():
            reber_text += char
            
    reber_text_long += next_letter + reber_text + next_letter + 'E'
    if len(reber_text_long) > 40:
        reber_text_long = gen_reber()
    return reber_text_long

print(gen_reber())

def gen_reber_wrong():
    letters = ['B', 'P', 'T', 'S', 'X', 'V', 'E']
    word = gen_reber()
    word_len = len(word)
    index = random.randint(0, word_len-1)
    letter = letters[random.randint(0, 6)]
    while letter == word[index]:
        letter = letters[random.randint(0, 6)]
    return word[:index] + letter + word[index+1:]
    
print(gen_reber_wrong())

BPBPTVVEPE
BTBTSSXXSVVETE


In [3]:
def gen_dataset(length):
    data = []
    for i in range(length//2):
        data.append((gen_reber(), 1.0))
        data.append((gen_reber_wrong(), 0.0))
        
    random.shuffle(data)
    return data

train_set = gen_dataset(100000)
valid_set = gen_dataset(25000)
test_set  = gen_dataset(25000)

train_set[:5]

[('BPBPTVPXVVEPE', 1.0),
 ('BTBTSSXSETE', 1.0),
 ('BPBTXSEPE', 1.0),
 ('BTBTSXSETE', 1.0),
 ('BTBTXSETE', 1.0)]

In [4]:
bpe_tokenizer_model = tokenizers.models.BPE(unk_token='<unk>')
bpe_tokenizer = tokenizers.Tokenizer(bpe_tokenizer_model)
bpe_tokenizer.enable_padding(pad_id=0, pad_token='<pad>')
bpe_tokenizer.enable_truncation(max_length=40)
bpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.ByteLevel()
bpe_tokenizer_trainer = tokenizers.trainers.BpeTrainer(vocab_size=256, special_tokens=['<unk>', '<pad>'])
train_rebers = [text[0] for text in train_set]
bpe_tokenizer.train_from_iterator(train_rebers, bpe_tokenizer_trainer)

In [5]:
fields = ['src_token_ids', 'src_mask']
class NmtPair(namedtuple("NmtPairBase", fields)):
    def to(self, device):
        return NmtPair(self.src_token_ids.to(device), self.src_mask.to(device))

def collate_fn(batch):
    rebers = [reber[0] for reber in batch]
    targets = torch.tensor([[label[1]] for label in batch])
    src_encodings = bpe_tokenizer.encode_batch(rebers)
    src_token_ids = torch.tensor([enc.ids for enc in src_encodings])
    src_mask = torch.FloatTensor([enc.attention_mask for enc in src_encodings])
    inputs = NmtPair(src_token_ids, src_mask)
    return inputs, targets

train_loader = DataLoader(train_set, batch_size=64, collate_fn=collate_fn, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=64, collate_fn=collate_fn)
test_loader = DataLoader(test_set, batch_size=64, collate_fn=collate_fn)

In [6]:
def eval_model(model, metric, data_loader):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
        return metric.compute()
    
def train_model(model, optimizer, criterion, train_loader, valid_loader, metric, n_epochs=50, patience=10, factor=0.1):
    history = {'train_losses':[], 'train_metrics':[], 'valid_metrics':[]}
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, mode='min', factor=factor, patience=patience)
    for epoch in range(n_epochs):
        model.train()
        metric.reset()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            #print(y_pred[:5], y_batch[:5])
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        history['train_losses'].append(total_loss/len(train_loader))
        history['train_metrics'].append(metric.compute().item())
        val_score = eval_model(model, metric, valid_loader).item()
        history['valid_metrics'].append(val_score)
        scheduler.step(val_score)
        
        print(f'Epoch: {epoch+1}\tTrain loss: {history["train_losses"][-1]:.3f}\tTrain metrics: {history["train_metrics"][-1]:.3f}\tValid metrics: {history["valid_metrics"][-1]:.3f}')
        
    return history

In [7]:
for X, y in train_loader:
    print(X[0].shape)
    break

torch.Size([64, 5])


In [8]:
def attention(query, key, value):
    scores = query @ key.transpose(1, 2)
    weights = torch.softmax(scores, dim=-1)
    return weights @ value

class ReberModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=256, n_layers=2, pad_id=0, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.gru = nn.GRU(embed_dim, hidden_dim, n_layers, batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, 1)
        
    def forward(self, X):
        embeddings = self.embed(X[0])
        lengths = X[1].sum(dim=1)
        packed = pack_padded_sequence(embeddings, lengths=lengths.cpu(), batch_first=True, enforce_sorted=False)
        _outputs, hidden_states = self.gru(packed)
        return self.output(hidden_states[-1])


In [10]:
model_1 = ReberModel(256).to(device)

optimizer = torch.optim.NAdam(params=model_1.parameters())
bxentropy = nn.BCEWithLogitsLoss()
metric = torchmetrics.Accuracy(task='binary').to(device)

history = train_model(model_1, optimizer, bxentropy, train_loader, valid_loader, metric, 10)

Epoch: 1	Train loss: 0.045	Train metrics: 0.984	Valid metrics: 0.997
Epoch: 2	Train loss: 0.004	Train metrics: 0.999	Valid metrics: 1.000
Epoch: 3	Train loss: 0.000	Train metrics: 1.000	Valid metrics: 1.000
Epoch: 4	Train loss: 0.000	Train metrics: 1.000	Valid metrics: 1.000
Epoch: 5	Train loss: 0.000	Train metrics: 1.000	Valid metrics: 1.000
Epoch: 6	Train loss: 0.000	Train metrics: 1.000	Valid metrics: 1.000
Epoch: 7	Train loss: 0.000	Train metrics: 1.000	Valid metrics: 1.000
Epoch: 8	Train loss: 0.000	Train metrics: 1.000	Valid metrics: 1.000
Epoch: 9	Train loss: 0.000	Train metrics: 1.000	Valid metrics: 1.000
Epoch: 10	Train loss: 0.000	Train metrics: 1.000	Valid metrics: 1.000


In [11]:
eval_model(model_1, metric, test_loader)

tensor(1., device='cuda:0')

In [12]:
model_1.eval()
with torch.no_grad():
    for X, y in train_loader:
        X = X.to(device)
        print((torch.sigmoid(model_1(X))>0.5).long()[:5])
        print(y[:5])
        break

tensor([[1],
        [0],
        [1],
        [0],
        [0]], device='cuda:0')
tensor([[1.],
        [0.],
        [1.],
        [0.],
        [0.]])


In [16]:
print(test_set[:3])
batch, _ = collate_fn(test_set[:3])
#print(batch, _)
model_1.eval()
with torch.no_grad():
    X = batch.to(device)
    print((torch.sigmoid(model_1(X))>0.5).long())
    

[('BPBPTVVEPE', 1.0), ('BPBPVPSEPE', 1.0), ('BPBPTPVEPE', 0.0)]
tensor([[1],
        [1],
        [0]], device='cuda:0')
